# SQL in Data Pipelines

## Overview
Real data pipelines combine SQL and Python: extract from a source database, transform in Python/pandas, and load into a target database. This notebook covers the patterns that make pipelines reliable, efficient, and maintainable.

**The ELT/ETL loop in Python:**
```
1. Extract  — read_sql_query() with chunksize; push filter/agg into SQL
2. Transform — pandas operations; type coercion; deduplication; validation
3. Load      — to_sql() with if_exists='append'; custom upsert for incremental
4. Audit     — log row counts; record high-water marks; handle failures
```

**Key principles:**
- Push as much work as possible into SQL (WHERE, GROUP BY, aggregations) before pulling to Python
- Process large tables in chunks; never `fetchall()` a table with millions of rows
- Use upserts for incremental loads — not `if_exists='replace'`
- Commit in batches of 10,000-100,000 rows; not per-row and not in a single mega-transaction
- Log every load with source row count, transformed row count, and load timestamp

---

In [1]:
import sqlite3, pandas as pd
from sqlalchemy import create_engine, text
import datetime, hashlib

# ── Simulate source and target databases ─────────────────────────
source_engine = create_engine("sqlite:///:memory:", echo=False)
target_engine = create_engine("sqlite:///:memory:", echo=False)

with source_engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE raw_patients (
            patient_id TEXT, first_name TEXT, last_name TEXT,
            province TEXT, dob TEXT, sex TEXT,
            loaded_at TEXT DEFAULT (datetime('now'))
        )
    """))
    conn.execute(text("""
        INSERT INTO raw_patients(patient_id,first_name,last_name,province,dob,sex) VALUES
            ('P001','Aroha',  'Ngata',   'NB','1985-03-15','F'),
            ('P001','Aroha',  'Ngata',   'NB','1985-03-15','F'),  -- duplicate
            ('P002','Liam',   'Chen',    'ON','1990-07-22','M'),
            ('P003','Fatima', 'Rashid',  'BC','1978-11-05','F'),
            ('P004','James',  'MacLeod', 'NB','2001-01-30','M'),
            ('P005','Maria',  'Santos',  'QC','1995-06-18','F'),
            ('P006','David',  'Park',    'AB',NULL,         'M'),  -- missing dob
            ('P007','',       'Brown',   'ON','1999-12-01','M')    -- empty first_name
    """))

with target_engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE dim_patients (
            patient_id  TEXT PRIMARY KEY,
            first_name  TEXT NOT NULL,
            last_name   TEXT NOT NULL,
            province    TEXT NOT NULL,
            dob         TEXT,
            sex         TEXT NOT NULL,
            age_bucket  TEXT,
            loaded_at   TEXT NOT NULL,
            row_hash    TEXT NOT NULL
        )
    """))
    conn.execute(text("""
        CREATE TABLE etl_audit (
            run_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            table_name  TEXT NOT NULL,
            source_rows INTEGER,
            valid_rows  INTEGER,
            loaded_rows INTEGER,
            rejected_rows INTEGER,
            run_at      TEXT NOT NULL DEFAULT (datetime('now'))
        )
    """))

print("Source and target databases initialised")


Source and target databases initialised


---
## Extract, validate, and transform

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
import datetime

source_engine = create_engine("sqlite:///:memory:", echo=False)
target_engine = create_engine("sqlite:///:memory:", echo=False)

with source_engine.begin() as conn:
    conn.execute(text("CREATE TABLE raw_patients (patient_id TEXT, first_name TEXT, last_name TEXT, province TEXT, dob TEXT, sex TEXT, loaded_at TEXT DEFAULT (datetime('now')))"))
    conn.execute(text("INSERT INTO raw_patients(patient_id,first_name,last_name,province,dob,sex) VALUES ('P001','Aroha','Ngata','NB','1985-03-15','F'),('P001','Aroha','Ngata','NB','1985-03-15','F'),('P002','Liam','Chen','ON','1990-07-22','M'),('P003','Fatima','Rashid','BC','1978-11-05','F'),('P004','James','MacLeod','NB','2001-01-30','M'),('P005','Maria','Santos','QC','1995-06-18','F'),('P006','David','Park','AB',NULL,'M'),('P007','','Brown','ON','1999-12-01','M')"))

with target_engine.begin() as conn:
    conn.execute(text("CREATE TABLE dim_patients (patient_id TEXT PRIMARY KEY, first_name TEXT NOT NULL, last_name TEXT NOT NULL, province TEXT NOT NULL, dob TEXT, sex TEXT NOT NULL, age_bucket TEXT, loaded_at TEXT NOT NULL, row_hash TEXT NOT NULL)"))
    conn.execute(text("CREATE TABLE etl_audit (run_id INTEGER PRIMARY KEY AUTOINCREMENT, table_name TEXT NOT NULL, source_rows INTEGER, valid_rows INTEGER, loaded_rows INTEGER, rejected_rows INTEGER, run_at TEXT NOT NULL DEFAULT (datetime('now')))"))

import hashlib

def compute_hash(row):
    s = "|".join(str(row[c]) for c in ["patient_id","first_name","last_name","province","dob","sex"])
    return hashlib.md5(s.encode()).hexdigest()

def age_bucket(dob):
    if pd.isna(dob):
        return "unknown"
    try:
        birth_year = int(str(dob)[:4])
        age = 2024 - birth_year
        if age < 18:   return "<18"
        if age < 35:   return "18-34"
        if age < 50:   return "35-49"
        if age < 65:   return "50-64"
        return "65+"
    except:
        return "unknown"

# ── Extract ───────────────────────────────────────────────────────
with source_engine.connect() as conn:
    df_raw = pd.read_sql_query(text("SELECT * FROM raw_patients"), conn)
source_rows = len(df_raw)
print(f"Extracted {source_rows} rows from source")

# ── Deduplicate ───────────────────────────────────────────────────
df = df_raw.drop_duplicates(subset=["patient_id"], keep="last")
print(f"After dedup: {len(df)} rows")

# ── Validate ─────────────────────────────────────────────────────
mask_valid = (
    df["patient_id"].notna() &
    df["first_name"].notna() & (df["first_name"].str.strip() != "") &
    df["last_name"].notna() &
    df["province"].notna() &
    df["sex"].isin(["M","F","Other"])
)
df_valid    = df[mask_valid].copy()
df_rejected = df[~mask_valid].copy()
print(f"Valid: {len(df_valid)} | Rejected: {len(df_rejected)}")
if len(df_rejected):
    print("Rejected rows:")
    print(df_rejected[["patient_id","first_name","province"]].to_string(index=False))


Extracted 8 rows from source
After dedup: 7 rows
Valid: 6 | Rejected: 1
Rejected rows:
patient_id first_name province
      P007                  ON


---
## Upsert pattern and audit logging

In [3]:
import pandas as pd, hashlib
from sqlalchemy import create_engine, text
import datetime

source_engine = create_engine("sqlite:///:memory:", echo=False)
target_engine = create_engine("sqlite:///:memory:", echo=False)
with source_engine.begin() as conn:
    conn.execute(text("CREATE TABLE raw_patients (patient_id TEXT, first_name TEXT, last_name TEXT, province TEXT, dob TEXT, sex TEXT)"))
    conn.execute(text("INSERT INTO raw_patients VALUES ('P001','Aroha','Ngata','NB','1985-03-15','F'),('P001','Aroha','Ngata','NB','1985-03-15','F'),('P002','Liam','Chen','ON','1990-07-22','M'),('P003','Fatima','Rashid','BC','1978-11-05','F'),('P004','James','MacLeod','NB','2001-01-30','M'),('P005','Maria','Santos','QC','1995-06-18','F'),('P006','David','Park','AB',NULL,'M'),('P007','','Brown','ON','1999-12-01','M')"))
with target_engine.begin() as conn:
    conn.execute(text("CREATE TABLE dim_patients (patient_id TEXT PRIMARY KEY, first_name TEXT NOT NULL, last_name TEXT NOT NULL, province TEXT NOT NULL, dob TEXT, sex TEXT NOT NULL, age_bucket TEXT, loaded_at TEXT NOT NULL, row_hash TEXT NOT NULL)"))
    conn.execute(text("CREATE TABLE etl_audit (run_id INTEGER PRIMARY KEY AUTOINCREMENT, table_name TEXT NOT NULL, source_rows INTEGER, valid_rows INTEGER, loaded_rows INTEGER, rejected_rows INTEGER, run_at TEXT NOT NULL DEFAULT (datetime('now')))"))

def compute_hash(row):
    s = "|".join(str(row[c]) for c in ["patient_id","first_name","last_name","province","dob","sex"])
    return hashlib.md5(s.encode()).hexdigest()

def age_bucket(dob):
    if pd.isna(dob): return "unknown"
    try:
        age = 2024 - int(str(dob)[:4])
        if age < 18: return "<18"
        if age < 35: return "18-34"
        if age < 50: return "35-49"
        if age < 65: return "50-64"
        return "65+"
    except: return "unknown"

# Rebuild from previous cell
with source_engine.connect() as conn:
    df_raw = pd.read_sql_query(text("SELECT * FROM raw_patients"), conn)
df = df_raw.drop_duplicates(subset=["patient_id"], keep="last")
mask = (df["patient_id"].notna() & df["first_name"].notna() &
        (df["first_name"].str.strip() != "") & df["province"].notna() &
        df["sex"].isin(["M","F","Other"]))
df_valid = df[mask].copy()
df_valid["age_bucket"] = df_valid["dob"].apply(age_bucket)
df_valid["loaded_at"]  = datetime.datetime.utcnow().isoformat()
df_valid["row_hash"]   = df_valid.apply(compute_hash, axis=1)

# Upsert: INSERT OR REPLACE for SQLite
cols = ["patient_id","first_name","last_name","province","dob","sex","age_bucket","loaded_at","row_hash"]
placeholders = ", ".join([f":{c}" for c in cols])
upsert_sql = f"INSERT OR REPLACE INTO dim_patients ({', '.join(cols)}) VALUES ({placeholders})"

loaded = 0
chunk_size = 3
with target_engine.begin() as conn:
    for i in range(0, len(df_valid), chunk_size):
        chunk = df_valid.iloc[i:i+chunk_size][cols]
        conn.execute(text(upsert_sql), chunk.to_dict(orient="records"))
        loaded += len(chunk)
        print(f"  Chunk {i//chunk_size + 1}: loaded {len(chunk)} rows (total so far: {loaded})")

# Audit log
with target_engine.begin() as conn:
    conn.execute(text(
        "INSERT INTO etl_audit(table_name,source_rows,valid_rows,loaded_rows,rejected_rows) "
        "VALUES (:t,:s,:v,:l,:r)"
    ), {"t": "dim_patients", "s": len(df_raw), "v": len(df_valid),
        "l": loaded, "r": len(df) - len(df_valid)})

with target_engine.connect() as conn:
    audit  = pd.read_sql_query(text("SELECT * FROM etl_audit"), conn)
    result = pd.read_sql_query(text("SELECT patient_id,first_name,province,age_bucket FROM dim_patients ORDER BY patient_id"), conn)

print()
print("Audit log:")
print(audit[["table_name","source_rows","valid_rows","loaded_rows","rejected_rows"]].to_string(index=False))
print()
print("Loaded dim_patients:")
print(result.to_string(index=False))


  Chunk 1: loaded 3 rows (total so far: 3)
  Chunk 2: loaded 3 rows (total so far: 6)

Audit log:
  table_name  source_rows  valid_rows  loaded_rows  rejected_rows
dim_patients            8           6            6              1

Loaded dim_patients:
patient_id first_name province age_bucket
      P001      Aroha       NB      35-49
      P002       Liam       ON      18-34
      P003     Fatima       BC      35-49
      P004      James       NB      18-34
      P005      Maria       QC      18-34
      P006      David       AB    unknown


C:\Users\saman\AppData\Local\Temp\ipykernel_24536\1323987257.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df_valid["loaded_at"]  = datetime.datetime.utcnow().isoformat()


---
## Common Pitfalls

**1. Loading entire source tables before filtering**
Pulling a full 100-million-row table into Python and filtering in pandas wastes memory, network bandwidth, and time. Push filtering into SQL WHERE clauses and aggregation into GROUP BY -- only transfer to Python what you actually need for the transformation step.

**2. Upsert with to_sql('replace') resetting auto-increment IDs**
`if_exists='replace'` drops and recreates the table, resetting sequences and losing indexes/constraints. For incremental loads, build a proper upsert using `INSERT OR REPLACE` (SQLite) or `INSERT ... ON CONFLICT DO UPDATE` (PostgreSQL).

**3. Not committing between chunks in large ETL loads**
Writing a million rows in a single transaction holds a transaction lock for the entire duration. Other readers and writers are blocked the entire time. Commit every N rows (10,000-100,000 is typical) to release the transaction lock periodically.

**4. Swallowing exceptions in batch loops without logging**
A bare `except: pass` inside a per-chunk processing loop silently discards failed chunks. In a 1,000-chunk load, 50 silent failures means the target table is missing 5% of data with no indication that anything went wrong. Always log failures with the chunk range and re-raise or write to a dead-letter table.

**5. Not validating data types before writing to a typed database**
A CSV with a 'cost' column that occasionally contains 'N/A' strings will fail when `to_sql` tries to write it to a REAL column. Always validate and coerce dtypes (`pd.to_numeric(df['cost'], errors='coerce')`) before writing, and decide what to do with rows that fail coercion.

**6. Relying on row order from SELECT without ORDER BY**
A `SELECT * FROM source_table` with no `ORDER BY` returns rows in undefined order -- which can vary between runs and database versions. Deterministic pipelines that depend on row order (windowing, deduplication, incremental loads keyed on position) must always include an explicit `ORDER BY` clause.


---
*sql_methods_library - Samantha McGarrigle*